# Comparación de Modelos de Regresión Vs HistGradientBoostingRegressor

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from joblib import load
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
import numpy as np

# Cargar los datos
ruta = r'C:\Users\jthow\iCloudDrive\Documents\3_Maestria_Estadistica_UNINORTE\3_Tercer_Semestre\Machine_Learning\tornados.csv.zip'
df = pd.read_csv(ruta)

# Definir X y y
X = df[['mag', 'slat', 'slon', 'elat', 'elon', 'len', 'wid', 'f1', 'f2', 'f3', 'f4', 'loss']]
y = df['inj']

# Manejar valores faltantes con imputación
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Convertir de nuevo a DataFrame para mantener los nombres de las columnas
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

# Dividir los datos en conjunto de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# Cargar los modelos entrenados
grid_ridge = load('grid_ridge.joblib')
grid_lasso = load('grid_lasso.joblib')
grid_knn = load('grid_knn.joblib')
grid_rf = load('grid_rf.joblib')
grid_xgb = load('grid_xgb.joblib')
grid_hgb = load('grid_hgb.joblib')
grid_lr = load('grid_lr.joblib')  # Cargar el modelo de regresión lineal

def calcular_metricas(model, X_test, y_test):
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred
    mape = mean_absolute_percentage_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    jb_p_value = jarque_bera(residuals)[1]
    lb_result = acorr_ljungbox(residuals, lags=[10], return_df=False)
    lb_p_value = lb_result['lb_pvalue'].iloc[0]

    return {
        "MAPE": f"{mape:.2f}",
        "RMSE": f"{rmse:.2f}",
        "R Cuadrado": f"{r2:.2f}",
        "Ljung-Box p-value": f"{lb_p_value:.2f}",
        "Jarque-Bera p-value": f"{jb_p_value:.2f}"
    }

# Calcular métricas
metricas_ridge = calcular_metricas(grid_ridge, X_test, y_test)
metricas_lasso = calcular_metricas(grid_lasso, X_test, y_test)
metricas_knn = calcular_metricas(grid_knn, X_test, y_test)
metricas_rf = calcular_metricas(grid_rf, X_test, y_test)
metricas_xgb = calcular_metricas(grid_xgb, X_test, y_test)
metricas_hgb = calcular_metricas(grid_hgb, X_test, y_test)  # No usar .best_estimator_
metricas_lr = calcular_metricas(grid_lr, X_test, y_test)  # Calcular métricas para el modelo de regresión lineal

# Crear DataFrames individuales
resultados_ridge = pd.DataFrame([metricas_ridge], index=["Ridge"])
resultados_lasso = pd.DataFrame([metricas_lasso], index=["Lasso"])
resultados_knn = pd.DataFrame([metricas_knn], index=["KNN"])
resultados_rf = pd.DataFrame([metricas_rf], index=["Random Forest"])
resultados_xgb = pd.DataFrame([metricas_xgb], index=["XGBoost"])
resultados_hgb = pd.DataFrame([metricas_hgb], index=["HistGradientBoosting"])
resultados_lr = pd.DataFrame([metricas_lr], index=["Linear Regression"])

# Combinar todos los DataFrames en uno solo
resultados_combinados = pd.concat([resultados_ridge, 
                                   resultados_lasso, 
                                   resultados_knn, 
                                   resultados_rf, 
                                   resultados_xgb, 
                                   resultados_hgb,
                                   resultados_lr])  # Incluir resultados de regresión lineal

# Mostrar la tabla combinada
display(resultados_combinados)


,MAPE,RMSE,R Cuadrado,Ljung-Box p-value,Jarque-Bera p-value
Ridge,7992741276210967.00,18.52,0.35,0.92,0.00
Lasso,7731547238791432.00,18.56,0.35,0.93,0.00
KNN,28846114506776.71,22.97,0.00,0.83,0.00
Random Forest,2184012427984590.00,17.43,0.43,0.27,0.00
XGBoost,2260117286289408.00,19.63,0.27,0.91,0.00
HistGradientBoosting,2378573422545617.50,19.35,0.29,0.94,0.00
Linear Regression,7998295564932685.00,18.52,0.35,0.92,0.00
